# Arriendos Gran Concepción — Portal Inmobiliario

Descarga, revisión de calidad y procesamiento de avisos de arriendo (casas y departamentos).

## 1. Setup

In [52]:
import sys
sys.executable

'/Users/ianlopezcandia/.pyenv/versions/3.10.0/bin/python'

In [53]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import COMMUNES, TYPES
from src.portal.crawler import run_crawl, RAW_DIR
from src.geocoding import update_cache

PROCESSED_DIR = ROOT / "data" / "processed"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Descarga
Elegir comunas y número máximo de páginas. Las búsquedas ya descargadas hoy se saltan.
**Si solo quieres analizar datos ya descargados, sáltate esta sección.**

In [54]:
# Test run
# communes_to_run = {k: COMMUNES[k] for k in ["Concepción", "Talcahuano"]}
# MAX_PAGES = 3

# Full run
communes_to_run = COMMUNES
MAX_PAGES = 50

In [55]:
with ThreadPoolExecutor(max_workers=1) as executor:
    run_dir = executor.submit(run_crawl, communes=communes_to_run, max_pages=MAX_PAGES).result()

print("Guardados en:", run_dir)

Concepción - departamento
    page 1: 48 listings
    page 2: 48 listings
    page 3: 48 listings
    page 4: 48 listings
    page 5: 48 listings
    page 6: 48 listings
    page 7: 48 listings
    page 8: 48 listings
    page 9: 48 listings
    page 10: 48 listings
    page 11: 48 listings
    page 12: 48 listings
    page 13: 48 listings
    page 14: 48 listings
    page 15: 48 listings
    page 16: 48 listings
    page 17: 48 listings
    page 18: 48 listings
    page 19: 48 listings
    page 20: 48 listings
    page 21: 20 listings
  -> saved 980 listings

Concepción - casa
    page 1: 48 listings
    page 2: 40 listings
  -> saved 88 listings

Talcahuano - departamento
    page 1: 48 listings
    page 2: 6 listings
  -> saved 54 listings

Talcahuano - casa
    page 1: 29 listings
  -> saved 29 listings

Hualpén - departamento
    page 1: 31 listings
  -> saved 31 listings

Hualpén - casa
    page 1: 2 listings
  -> saved 2 listings

San Pedro de la Paz - departamento
    page 1: 4

## 3. Carga

In [56]:
# Latest run, or set a specific date: RUN = "2026-10-24"
RUN = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())[-1]
run_dir = RAW_DIR / RUN

files = sorted(run_dir.glob("*.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(f"Run {RUN}: {len(files)} files, {len(df)} listings")

Run 2026-10-24: 16 files, 1635 listings


In [57]:
df.pivot_table(index="commune", columns="property_type",
               values="id", aggfunc="count", fill_value=0)

property_type,casa,departamento
commune,,
Chiguayante,23,58
Concepción,88,980
Coronel,13,4
Hualpén,2,31
Hualqui,2,0
Penco,1,0
San Pedro de la Paz,60,278
Talcahuano,29,54
Tomé,7,5


## 4. Calidad

In [58]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})
quality[quality["missing"] > 0].sort_values("missing_%", ascending=False)

,missing,missing_%
total_m2,1617,98.9
broker,1404,85.9
usable_m2,53,3.2
bedrooms,17,1.0
price,10,0.6
bathrooms,8,0.5
neighborhood,7,0.4


## 5. Procesamiento
1. Unir y quitar duplicados
2. Filtrar avisos inválidos
3. Agregar coordenadas (solo se consultan direcciones nuevas; el resto sale del caché)
4. Guardar en `data/processed/`

In [59]:
# 5.1 + 5.2  Unir y filtrar
consolidated = (
    df.drop_duplicates(subset=["id"])
      .drop(columns=["total_m2", "broker"], errors="ignore")
      .dropna(subset=["price"])          # without price the listing is useless
)

print(f"Rows: {len(df)} -> {len(consolidated)}")

Rows: 1635 -> 1625


In [60]:
# 5.3a  PRUEBA: geolocalizar solo 30 direcciones (ejecutar una vez y revisar)
cache = update_cache(consolidated.head(30))
cache[["location", "precision", "lat", "lon"]].tail(10)

In cache: 0 | new to geocode: 28 (about 1 min)


/Users/ianlopezcandia/Workspace/cdatos /arriendo-gran-concepcion/src/geocoding.py:148: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


  25/28 done
  28/28 done


,location,precision,lat,lon
18,"Avenida O'Higgins 1935, Chiguayante, Chile, Al...",neighborhood,-36.920549,-73.034066
19,"ALTO CASTAÑO, Chiguayante Sur, Chiguayante",neighborhood,-36.931095,-73.030801
20,"Vía Del Valle 89, Concepción, Chiguayante, Vil...",address,-36.879264,-73.035966
21,"Cosme Churruca, Concepción, Lomas De San André...",street,-36.792452,-73.049079
22,"Plaza Juan Bosco, Barrio Oriente - Las Tres Pa...",street,-36.818085,-73.043176
23,"García Hurtado De Mendoza 680, Concepción, Lom...",address,-36.791606,-73.056981
24,"Cochrane - Chacabuco, Centro De Concepción, Co...",neighborhood,-36.826701,-73.048530
25,"Cosme Churruca 372, Concepción, Lomas De San A...",address,-36.787984,-73.054359
26,"Lomas De San Sebastián, Concepción",street,-36.780227,-73.049092
27,"Víctor Lamas 1038, Concepción, Concepción",address,-36.830075,-73.042150


In [61]:
## borra el cache de la geolocalización para que se vuelva a intentar la geolocalización de las direcciones que no se encontraron
##from src.geocoding import forget_not_found
##forget_not_found()

In [62]:
# 5.3b  Geolocalizar todo (la primera vez tarda 20-40 min)
cache = update_cache(consolidated)
cache["precision"].value_counts()

In cache: 28 | new to geocode: 1141 (about 29 min)
  25/1141 done
  50/1141 done
  75/1141 done
  100/1141 done
  125/1141 done
  150/1141 done
  175/1141 done
  200/1141 done
  225/1141 done
  250/1141 done
  275/1141 done
  300/1141 done
  325/1141 done
  350/1141 done
  375/1141 done
  400/1141 done
  425/1141 done
  450/1141 done
  475/1141 done
  500/1141 done
  525/1141 done
  550/1141 done
  575/1141 done
  600/1141 done
  625/1141 done
  650/1141 done
  675/1141 done
  700/1141 done
  725/1141 done
  750/1141 done
  775/1141 done
  800/1141 done
  825/1141 done
  850/1141 done
  875/1141 done
  900/1141 done
  925/1141 done
  950/1141 done
  975/1141 done
  1000/1141 done
  1025/1141 done
  1050/1141 done
  1075/1141 done
  1100/1141 done
  1125/1141 done
  1141/1141 done


precision
address         596
street          342
neighborhood    183
not_found        48
Name: count, dtype: int64

In [63]:
# 5.4  Unir coordenadas y guardar
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed = consolidated.merge(
    cache[["location", "lat", "lon", "precision"]],
    on="location", how="left",
)

output_file = PROCESSED_DIR / f"portal_{RUN}.csv"
processed.to_csv(output_file, index=False, encoding="utf-8")

print(f"{len(processed)} listings, {processed['lat'].notna().sum()} with coordinates")
print("Saved:", output_file.relative_to(ROOT))

1625 listings, 1568 with coordinates
Saved: data/processed/portal_2026-10-24.csv


In [64]:
# 5.5  Coordenadas por comuna y precisión
summary = (
    processed.assign(precision=processed["precision"].fillna("not_found"))
    .pivot_table(index="commune", columns="precision",
                 values="id", aggfunc="count", fill_value=0)
)
summary["%_with_coords"] = (
    100 * (1 - summary.get("not_found", 0) / summary.sum(axis=1))
).round(1)
summary

precision,address,neighborhood,not_found,street,%_with_coords
commune,,,,,
Chiguayante,41,20,1,19,98.8
Concepción,600,133,17,312,98.4
Coronel,2,4,4,7,76.5
Hualpén,12,4,10,7,69.7
Hualqui,0,0,2,0,0.0
Penco,0,0,1,0,0.0
San Pedro de la Paz,159,56,19,101,94.3
Talcahuano,47,6,3,27,96.4
Tomé,1,1,0,9,100.0


In [65]:
# 5.6  Direcciones sin coordenadas
missing = processed[processed["lat"].isna()]
missing[["commune", "location", "neighborhood"]].drop_duplicates().head(20)

,commune,location,neighborhood
17,Chiguayante,"CONDOMINIO LAS AZUCENAS, Chiguayante",CONDOMINIO LAS AZUCENAS
36,Concepción,"Reserva Nonguen, Concepción",Reserva Nonguen
43,Concepción,"Av. Roosevelt 1666, Concepción, Concepción",Concepción
112,Coronel,"Villa Esperanza, Coronel",Villa Esperanza
123,Coronel,"Pje. Frutillar, Coronel, Coronel",Coronel
124,Hualpén,"Toulouse, Hualpén, Hualpén",Hualpén
125,Hualpén,"Pje. Arezzo, Hualpén, Hualpén",Hualpén
126,Hualqui,"HUALQUI, Hualqui",HUALQUI
127,Hualqui,Hualqui,NaN
128,Penco,"Condominio Terrazas De Landa, Penco, Penco",Penco


## 6. Mapa de verificación
Solo para comprobar que los puntos caen donde deben.

In [66]:
import plotly.express as px

points = processed.dropna(subset=["lat"])
fig = px.scatter_map(
    points, lat="lat", lon="lon", color="commune",
    hover_data=["price", "currency", "bedrooms", "usable_m2", "neighborhood", "precision"],
    zoom=10, height=650,
)
fig.show()

In [ ]:

pts = processed.dropna(subset=["lat"]).copy()

# Center of each commune = median of its points (robust to a few wrong ones)
centers = pts.groupby("commune")[["lat", "lon"]].median()
pts = pts.join(centers, on="commune", rsuffix="_center")

# Approximate distance in km (good enough at this scale)
pts["km_from_center"] = np.sqrt(
    ((pts["lat"] - pts["lat_center"]) * 111) ** 2
    + ((pts["lon"] - pts["lon_center"]) * 111 * np.cos(np.radians(pts["lat"]))) ** 2
).round(1)

(pts.sort_values("km_from_center", ascending=False)
    [["commune", "location", "precision", "km_from_center", "lat", "lon"]]
    .head(20))

,commune,location,precision,km_from_center,lat,lon
117,Coronel,"Valle Mitrinhue, Coronel, Coronel",street,15.7,-36.980118,-72.967104
119,Coronel,"Valle Mitrinhue, Coronel, Coronel",street,15.7,-36.980118,-72.967104
118,Coronel,"COD40008 Valle Mitrinhue, Camino A Santa Juana...",neighborhood,14.5,-37.054672,-72.984795
111,Coronel,"Camino A Santa Juana, Coronel",street,14.5,-37.054672,-72.984795
135,San Pedro de la Paz,"Camino A Santa Juana, San Pedro De La Paz",street,12.3,-36.951426,-73.049250
1624,Tomé,"Av. Werner 2500, Tome, Tomé, Ciudad De Tomé, Tomé",address,9.2,-36.609562,-72.964808
1622,Tomé,"Werner, 2500, Ciudad De Tomé, Tomé",street,8.9,-36.605335,-72.969575
1297,San Pedro de la Paz,"Concepción, Villa San Pedro - Bayona, San Pedr...",street,7.2,-36.905169,-73.144816
1440,San Pedro de la Paz,"Los Mañios 6095, San Pedro De La Paz, Portal S...",address,7.0,-36.903396,-73.144057
1541,San Pedro de la Paz,"Los Mañios 6095, San Pedro De La Paz, Portal S...",address,7.0,-36.903396,-73.144057
